# Claim data analysis (Snowflake, read-only)

One row = one **claim line** (`ClaimNo` + `ClaimLineNo`). One patient and one claim number can have many lines. **Read-only**: `SELECT` / `DESCRIBE` only.

**Cell format:** SQL cells so each result has Table / Chart / Pivot and the **download** button.

**Grain to confirm first**
- Unique **lines** should match `ROW_COUNT` (pair `ClaimNo`, `ClaimLineNo`).
- Unique **claims** (`ClaimNo`) will be smaller.
- Unique **patients** should line up with Census over time; this notebook only counts what is on Claims.

**High-value fields:** `ClaimStatus`, `ProcedureCode` (CPT/HCPCS), `DiagnosisType` + `DiagnosisCode`, provider (`Provider NPI`), dates of service, optional `DRGCode` / `REVCode`.

**ClinicalNotes on claims:** treat like clinical-note text — unique texts + **duplicates**, previews only. Do **not** run full-table word explode / TF-IDF on tens of millions of rows.

## 1. Active session

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

session = get_active_session()
session

## 2. Config (change names ONLY here)

`Place of Service` and `Provider NPI` have spaces. Keep `QUOTE_COLUMNS = True` unless `DESCRIBE` shows simple uppercase names.

In [ ]:
DATABASE_NAME = "ATTR"
SCHEMA_NAME = "PUBLIC"
TABLE_NAME = "CLAIM"   # try CLAIMS, MEDICAL_CLAIM, CLAIM_LINE

QUOTE_DATABASE = False
QUOTE_SCHEMA = False
QUOTE_TABLE = False
QUOTE_COLUMNS = True

COL = {
    "patient_id": "Member/PatientId",
    "encounter_id": "EncounterId/VisitId",
    "claim_no": "ClaimNo",
    "claim_line_no": "ClaimLineNo",
    "claim_status": "ClaimStatus",
    "date_incurred_from": "DateIncurredFrom",
    "date_incurred_to": "DateIncurredTo",
    "place_of_service": "Place of Service",
    "procedure_code": "ProcedureCode",
    "mod1": "ProcedureModifier1",
    "mod2": "ProcedureModifier2",
    "mod3": "ProcedureModifier3",
    "diagnosis_type": "DiagnosisType",
    "diagnosis_code": "DiagnosisCode",
    "provider_npi": "Provider NPI",
    "provider_name": "ProviderName",
    "provider_type": "ProviderType",
    "specialty_code": "SpecialtyCode",
    "specialty_name": "SpecialtyName",
    "provider_tax_id": "ProviderTaxId",
    "drg_code": "DRGCode",
    "rev_code": "REVCode",
    "other_dx_9": "OtherDiagnosisCodes9",
    "other_dx_10": "OtherDiagnosisCodes10",
    "from_date": "FromDate",
    "to_date": "ToDate",
    "clinical_notes": "ClinicalNotes",
}


def sf_ident(name, quoted):
    if quoted:
        return '"' + str(name).replace('"', '""') + '"'
    return str(name)


def col(key):
    return sf_ident(COL[key], QUOTE_COLUMNS)


DB = sf_ident(DATABASE_NAME, QUOTE_DATABASE)
T = ".".join(
    [
        DB,
        sf_ident(SCHEMA_NAME, QUOTE_SCHEMA),
        sf_ident(TABLE_NAME, QUOTE_TABLE),
    ]
)

C_PT = col("patient_id")
C_ENC = col("encounter_id")
C_CLM = col("claim_no")
C_LINE = col("claim_line_no")
C_STATUS = col("claim_status")
C_DOS_FROM = col("date_incurred_from")
C_DOS_TO = col("date_incurred_to")
C_POS = col("place_of_service")
C_CPT = col("procedure_code")
C_MOD1 = col("mod1")
C_MOD2 = col("mod2")
C_MOD3 = col("mod3")
C_DXTYPE = col("diagnosis_type")
C_DX = col("diagnosis_code")
C_NPI = col("provider_npi")
C_PNAME = col("provider_name")
C_PTYPE = col("provider_type")
C_SPEC = col("specialty_code")
C_SPECNAME = col("specialty_name")
C_TAX = col("provider_tax_id")
C_DRG = col("drg_code")
C_REV = col("rev_code")
C_DX9 = col("other_dx_9")
C_DX10 = col("other_dx_10")
C_FROM = col("from_date")
C_TO = col("to_date")
C_NOTE = col("clinical_notes")

for name, value in [
    ("DB", DB),
    ("T", T),
    ("C_PT", C_PT),
    ("C_ENC", C_ENC),
    ("C_CLM", C_CLM),
    ("C_LINE", C_LINE),
    ("C_STATUS", C_STATUS),
    ("C_DOS_FROM", C_DOS_FROM),
    ("C_DOS_TO", C_DOS_TO),
    ("C_POS", C_POS),
    ("C_CPT", C_CPT),
    ("C_DXTYPE", C_DXTYPE),
    ("C_DX", C_DX),
    ("C_NPI", C_NPI),
    ("C_PTYPE", C_PTYPE),
    ("C_SPEC", C_SPEC),
    ("C_DRG", C_DRG),
    ("C_REV", C_REV),
    ("C_FROM", C_FROM),
    ("C_TO", C_TO),
    ("C_NOTE", C_NOTE),
]:
    print(f"{name} = {value}")

## 3. Find the table (only if the name or schema is wrong)

In [ ]:
SELECT
    CURRENT_ROLE() AS ROLE,
    CURRENT_WAREHOUSE() AS WAREHOUSE,
    CURRENT_DATABASE() AS DATABASE,
    CURRENT_SCHEMA() AS SCHEMA;

In [ ]:
SELECT
    TABLE_CATALOG,
    TABLE_SCHEMA,
    TABLE_NAME,
    ROW_COUNT,
    BYTES
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
  AND (
        UPPER(TABLE_NAME) LIKE '%CLAIM%'
     OR UPPER(TABLE_NAME) LIKE '%ADJUDIC%'
  )
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## 4. Table shape and first 10 rows

Notes are previewed (200 characters) so the grid stays usable.

In [ ]:
DESCRIBE TABLE {{T}};

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    {{C_ENC}} AS ENCOUNTER_ID,
    {{C_CLM}} AS CLAIM_NO,
    {{C_LINE}} AS CLAIM_LINE_NO,
    {{C_STATUS}} AS CLAIM_STATUS,
    {{C_DOS_FROM}} AS DATE_INCURRED_FROM,
    {{C_DOS_TO}} AS DATE_INCURRED_TO,
    {{C_POS}} AS PLACE_OF_SERVICE,
    {{C_CPT}} AS PROCEDURE_CODE,
    {{C_DXTYPE}} AS DIAGNOSIS_TYPE,
    {{C_DX}} AS DIAGNOSIS_CODE,
    {{C_NPI}} AS PROVIDER_NPI,
    {{C_FROM}} AS FROM_DATE,
    {{C_TO}} AS TO_DATE,
    LEFT({{C_NOTE}}::STRING, 200) AS CLINICAL_NOTES_PREVIEW
FROM {{T}}
LIMIT 10;

## 5. Volume and uniqueness (grain)

If `EXTRA_ROWS_VS_UNIQUE_CLAIM_LINES` is not 0, the same claim line is duplicated.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_ENC}}) AS UNIQUE_ENCOUNTERS,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIM_NUMBERS,
    COUNT(DISTINCT {{C_CLM}} || '|' || {{C_LINE}}::STRING) AS UNIQUE_CLAIM_LINES,
    COUNT(*) - COUNT(DISTINCT {{C_CLM}} || '|' || {{C_LINE}}::STRING) AS EXTRA_ROWS_VS_UNIQUE_CLAIM_LINES,
    COUNT(DISTINCT {{C_CPT}}) AS UNIQUE_PROCEDURE_CODES,
    COUNT(DISTINCT {{C_DX}}) AS UNIQUE_DIAGNOSIS_CODES,
    COUNT(DISTINCT {{C_NPI}}) AS UNIQUE_PROVIDER_NPIS,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_CLM}}), 0), 2) AS AVG_LINES_PER_CLAIM,
    ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT {{C_PT}}), 0), 2) AS AVG_LINES_PER_PATIENT
FROM {{T}};

## 6. Completeness (nulls)

Required per dictionary: patient (or encounter), claim number, line, dates incurred, procedure, diagnosis type/code, NPI, FromDate, ToDate. Status, POS, modifiers, specialty, DRG, REV, other DX, notes are optional.

In [ ]:
SELECT
    COUNT(*) AS ROW_COUNT,
    SUM(IFF({{C_PT}} IS NULL, 1, 0)) AS NULL_PATIENT_ID,
    SUM(IFF({{C_ENC}} IS NULL, 1, 0)) AS NULL_ENCOUNTER_ID,
    SUM(IFF({{C_CLM}} IS NULL, 1, 0)) AS NULL_CLAIM_NO,
    SUM(IFF({{C_LINE}} IS NULL, 1, 0)) AS NULL_CLAIM_LINE_NO,
    SUM(IFF({{C_STATUS}} IS NULL, 1, 0)) AS NULL_CLAIM_STATUS,
    SUM(IFF({{C_DOS_FROM}} IS NULL, 1, 0)) AS NULL_DATE_INCURRED_FROM,
    SUM(IFF({{C_DOS_TO}} IS NULL, 1, 0)) AS NULL_DATE_INCURRED_TO,
    SUM(IFF({{C_POS}} IS NULL, 1, 0)) AS NULL_PLACE_OF_SERVICE,
    SUM(IFF({{C_CPT}} IS NULL, 1, 0)) AS NULL_PROCEDURE_CODE,
    SUM(IFF({{C_DXTYPE}} IS NULL, 1, 0)) AS NULL_DIAGNOSIS_TYPE,
    SUM(IFF({{C_DX}} IS NULL, 1, 0)) AS NULL_DIAGNOSIS_CODE,
    SUM(IFF({{C_NPI}} IS NULL, 1, 0)) AS NULL_PROVIDER_NPI,
    SUM(IFF({{C_DRG}} IS NULL OR TRIM({{C_DRG}}::STRING) IN ('', '0'), 1, 0)) AS NULL_OR_ZERO_DRG,
    SUM(IFF({{C_REV}} IS NULL OR TRIM({{C_REV}}::STRING) IN ('', '0'), 1, 0)) AS NULL_OR_ZERO_REV,
    SUM(IFF({{C_FROM}} IS NULL, 1, 0)) AS NULL_FROM_DATE,
    SUM(IFF({{C_TO}} IS NULL, 1, 0)) AS NULL_TO_DATE,
    SUM(IFF({{C_NOTE}} IS NULL OR TRIM({{C_NOTE}}::STRING) = '', 1, 0)) AS NULL_OR_BLANK_NOTES
FROM {{T}};

## 7. ClaimStatus — unique values with occurrences

In [ ]:
SELECT
    {{C_STATUS}} AS CLAIM_STATUS,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIMS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

## 8. ProcedureCode (CPT/HCPCS) — unique codes with occurrences

One row per stored procedure code. `LINE_COUNT` is how often that code appears on a claim line.

In [ ]:
SELECT
    {{C_CPT}} AS PROCEDURE_CODE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIMS,
    COUNT(DISTINCT {{C_NPI}}) AS UNIQUE_NPIS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    SUM(IFF({{C_MOD1}} IS NOT NULL AND TRIM({{C_MOD1}}::STRING) <> '', 1, 0)) AS LINES_WITH_MOD1,
    SUM(IFF({{C_MOD2}} IS NOT NULL AND TRIM({{C_MOD2}}::STRING) <> '', 1, 0)) AS LINES_WITH_MOD2,
    SUM(IFF({{C_MOD3}} IS NOT NULL AND TRIM({{C_MOD3}}::STRING) <> '', 1, 0)) AS LINES_WITH_MOD3,
    COUNT(*) AS LINE_COUNT
FROM {{T}};

In [ ]:
SELECT
    {{C_MOD1}} AS PROCEDURE_MODIFIER1,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_CPT}}) AS UNIQUE_PROCEDURE_CODES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

## 9. Diagnosis — type, unique codes, occurrences

`DiagnosisType` should be ICD-9 vs ICD-10 (or mixed). `diagnosis_code_counts` is the unique primary DX list with counts. `diagnosis_code_by_type` keeps type and code together so the same numeric-looking code is not mixed across ICD versions.

In [ ]:
SELECT
    {{C_DXTYPE}} AS DIAGNOSIS_TYPE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_DX}}) AS UNIQUE_DIAGNOSIS_CODES,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    {{C_DX}} AS DIAGNOSIS_CODE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_DXTYPE}}) AS DISTINCT_DIAGNOSIS_TYPES,
    LISTAGG(DISTINCT {{C_DXTYPE}}::STRING, ' | ') AS DIAGNOSIS_TYPES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    {{C_DXTYPE}} AS DIAGNOSIS_TYPE,
    {{C_DX}} AS DIAGNOSIS_CODE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1, 2
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    SUM(IFF({{C_DX9}} IS NOT NULL AND TRIM({{C_DX9}}::STRING) NOT IN ('', '0'), 1, 0)) AS LINES_WITH_OTHER_DX9,
    SUM(IFF({{C_DX10}} IS NOT NULL AND TRIM({{C_DX10}}::STRING) NOT IN ('', '0'), 1, 0)) AS LINES_WITH_OTHER_DX10,
    COUNT(*) AS LINE_COUNT
FROM {{T}};

## 10. Place of Service, provider, specialty, DRG, REV

In [ ]:
SELECT
    {{C_POS}} AS PLACE_OF_SERVICE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_CPT}}) AS UNIQUE_PROCEDURE_CODES,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    {{C_PTYPE}} AS PROVIDER_TYPE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_NPI}}) AS UNIQUE_NPIS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    {{C_SPEC}} AS SPECIALTY_CODE,
    {{C_SPECNAME}} AS SPECIALTY_NAME,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_NPI}}) AS UNIQUE_NPIS,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1, 2
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    {{C_NPI}} AS PROVIDER_NPI,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIMS,
    COUNT(DISTINCT {{C_PNAME}}) AS DISTINCT_PROVIDER_NAMES,
    LISTAGG(DISTINCT LEFT({{C_PNAME}}::STRING, 80), ' | ') AS PROVIDER_NAMES
FROM {{T}}
WHERE {{C_NPI}} IS NOT NULL
GROUP BY 1
ORDER BY LINE_COUNT DESC
LIMIT 50;

In [ ]:
SELECT
    {{C_NPI}} AS PROVIDER_NPI,
    COUNT(DISTINCT {{C_PNAME}}) AS UNIQUE_NAMES,
    COUNT(*) AS LINE_COUNT,
    LISTAGG(DISTINCT LEFT({{C_PNAME}}::STRING, 80), ' | ') AS NAME_VARIATIONS
FROM {{T}}
WHERE {{C_NPI}} IS NOT NULL
GROUP BY 1
HAVING COUNT(DISTINCT {{C_PNAME}}) > 1
ORDER BY UNIQUE_NAMES DESC, LINE_COUNT DESC
LIMIT 200;

In [ ]:
SELECT
    {{C_DRG}} AS DRG_CODE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

In [ ]:
SELECT
    {{C_REV}} AS REV_CODE,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY LINE_COUNT DESC;

## 11. Service dates and encounter dates

`DateIncurredFrom` / `DateIncurredTo` = dates of service on the claim. `FromDate` / `ToDate` = encounter window (often the same as FromDate). Recency uses **DateIncurredFrom** vs today.

In [ ]:
SELECT
    CURRENT_DATE() AS TODAY,
    MIN({{C_DOS_FROM}}) AS MIN_DATE_INCURRED_FROM,
    MAX({{C_DOS_FROM}}) AS MAX_DATE_INCURRED_FROM,
    MIN({{C_DOS_TO}}) AS MIN_DATE_INCURRED_TO,
    MAX({{C_DOS_TO}}) AS MAX_DATE_INCURRED_TO,
    MIN({{C_FROM}}) AS MIN_FROM_DATE,
    MAX({{C_FROM}}) AS MAX_FROM_DATE,
    SUM(IFF({{C_DOS_FROM}}::DATE > {{C_DOS_TO}}::DATE, 1, 0)) AS LINES_FROM_AFTER_TO,
    SUM(IFF({{C_DOS_FROM}}::DATE > CURRENT_DATE(), 1, 0)) AS FUTURE_INCURRED_FROM,
    SUM(IFF({{C_DOS_FROM}}::DATE IS DISTINCT FROM {{C_FROM}}::DATE, 1, 0)) AS LINES_INCURRED_FROM_NE_FROMDATE
FROM {{T}};

In [ ]:
SELECT
    YEAR({{C_DOS_FROM}}) AS SERVICE_YEAR,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIMS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_LINES
FROM {{T}}
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH bucketed AS (
    SELECT
        CASE
            WHEN {{C_DOS_FROM}} IS NULL THEN 90
            WHEN {{C_DOS_FROM}}::DATE > CURRENT_DATE() THEN 80
            WHEN DATEDIFF('day', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 30 THEN 1
            WHEN DATEDIFF('day', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 90 THEN 2
            WHEN DATEDIFF('day', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 180 THEN 3
            WHEN DATEDIFF('day', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 365 THEN 4
            WHEN DATEDIFF('year', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 2 THEN 5
            WHEN DATEDIFF('year', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 5 THEN 6
            WHEN DATEDIFF('year', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 7 THEN 7
            WHEN DATEDIFF('year', {{C_DOS_FROM}}::DATE, CURRENT_DATE()) <= 10 THEN 8
            ELSE 9
        END AS SORT_ORDER,
        {{C_PT}} AS PATIENT_ID
    FROM {{T}}
)
SELECT
    SORT_ORDER,
    CASE SORT_ORDER
        WHEN 1 THEN '0-30 days ago'
        WHEN 2 THEN '31-90 days ago'
        WHEN 3 THEN '91-180 days ago'
        WHEN 4 THEN '181-365 days ago'
        WHEN 5 THEN '1-2 years ago'
        WHEN 6 THEN '3-5 years ago'
        WHEN 7 THEN '6-7 years ago'
        WHEN 8 THEN '8-10 years ago'
        WHEN 9 THEN 'More than 10 years ago'
        WHEN 80 THEN 'Future (after today)'
        ELSE 'Unknown (missing date)'
    END AS DATE_RANGE,
    COUNT(*) AS ROW_COUNT,
    COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENTS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_ROWS
FROM bucketed
GROUP BY 1, 2
ORDER BY SORT_ORDER;

## 12. Lines per claim and claims per patient

In [ ]:
WITH per_claim AS (
    SELECT
        {{C_CLM}} AS CLAIM_NO,
        COUNT(*) AS LINE_COUNT
    FROM {{T}}
    WHERE {{C_CLM}} IS NOT NULL
    GROUP BY 1
)
SELECT
    LINE_COUNT AS LINES_ON_CLAIM,
    COUNT(*) AS NUMBER_OF_CLAIMS,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS PCT_OF_CLAIMS
FROM per_claim
GROUP BY 1
ORDER BY 1;

In [ ]:
WITH per_pt AS (
    SELECT
        {{C_PT}} AS PATIENT_ID,
        COUNT(*) AS LINE_COUNT,
        COUNT(DISTINCT {{C_CLM}}) AS CLAIM_COUNT
    FROM {{T}}
    WHERE {{C_PT}} IS NOT NULL
    GROUP BY 1
)
SELECT
    LINE_COUNT AS LINES_PER_PATIENT,
    COUNT(*) AS NUMBER_OF_PATIENTS
FROM per_pt
GROUP BY 1
ORDER BY 1;

In [ ]:
SELECT
    {{C_PT}} AS PATIENT_ID,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIMS,
    COUNT(DISTINCT {{C_CPT}}) AS UNIQUE_PROCEDURE_CODES,
    COUNT(DISTINCT {{C_DX}}) AS UNIQUE_DIAGNOSIS_CODES,
    MIN({{C_DOS_FROM}}) AS FIRST_SERVICE_DATE,
    MAX({{C_DOS_FROM}}) AS LAST_SERVICE_DATE
FROM {{T}}
WHERE {{C_PT}} IS NOT NULL
GROUP BY 1
ORDER BY LINE_COUNT DESC
LIMIT 25;

In [ ]:
SELECT
    {{C_CLM}} AS CLAIM_NO,
    {{C_LINE}} AS CLAIM_LINE_NO,
    {{C_PT}} AS PATIENT_ID,
    {{C_STATUS}} AS CLAIM_STATUS,
    {{C_DOS_FROM}} AS DATE_INCURRED_FROM,
    {{C_CPT}} AS PROCEDURE_CODE,
    {{C_DX}} AS DIAGNOSIS_CODE,
    {{C_DXTYPE}} AS DIAGNOSIS_TYPE,
    {{C_NPI}} AS PROVIDER_NPI,
    {{C_POS}} AS PLACE_OF_SERVICE
FROM {{T}}
WHERE {{C_CLM}} = (
        SELECT {{C_CLM}}
        FROM {{T}}
        WHERE {{C_CLM}} IS NOT NULL
        GROUP BY 1
        ORDER BY COUNT(*) DESC
        LIMIT 1
      )
ORDER BY {{C_LINE}};

## 13. ClinicalNotes on the claim (optional text)

Same rules as Clinical Note: unique texts + duplicates, **preview only**. No full-table word explode (slow on tens of millions of rows).

In [ ]:
SELECT
    COUNT(*) AS LINE_COUNT,
    SUM(IFF({{C_NOTE}} IS NULL OR TRIM({{C_NOTE}}::STRING) = '', 1, 0)) AS EMPTY_NOTE_LINES,
    COUNT(DISTINCT {{C_NOTE}}) AS UNIQUE_NOTE_TEXTS,
    ROUND(AVG(LENGTH({{C_NOTE}}::STRING)), 0) AS AVG_CHAR_LENGTH
FROM {{T}};

In [ ]:
SELECT
    LEFT({{C_NOTE}}::STRING, 300) AS NOTE_PREVIEW,
    LENGTH({{C_NOTE}}::STRING) AS NOTE_CHAR_LENGTH,
    COUNT(*) AS LINE_COUNT,
    COUNT(DISTINCT {{C_PT}}) AS UNIQUE_PATIENTS,
    COUNT(DISTINCT {{C_CLM}}) AS UNIQUE_CLAIMS
FROM {{T}}
WHERE {{C_NOTE}} IS NOT NULL
  AND TRIM({{C_NOTE}}::STRING) <> ''
GROUP BY {{C_NOTE}}
HAVING COUNT(*) > 1
ORDER BY LINE_COUNT DESC;

## Notes

- **Downloading:** use the download arrow on each SQL result grid.
- `procedure_code_counts` and `diagnosis_code_counts` can be large (tens of thousands of codes). That is expected; download if needed.
- Same NPI with several `ProviderName` spellings is listed in `npi_name_variations`.
- Run `config` before SQL cells.
- Linking to Census is a later join: `Member/PatientId` on both tables. This notebook stays on Claims only.